Классический прунинг (Han et al., 2015; OBD; OBS) исходит из идеи: сначала обучить плотную сеть, затем удалить «лишние» веса, затем дообучить. Это эффективный, но дорогой подход — требуется хранить и обучать полную модель, что для больших сетей становится узким местом.

Динамический прунинг (его также называют sparse training, dynamic sparse training, DST) предлагает альтернативный подход: разреженность поддерживается на протяжении всего обучения. Сеть никогда не существует в плотном виде; маска связей эволюционирует параллельно с весами. Это резко снижает требования к памяти и вычислениям и теоретически открывает путь к обучению гигантских разреженных моделей на ограниченном железе.

В этом пособии мы разберём ключевые методы динамического прунинга: SET, SNIP, GraSP, RigL и другие. По каждому методу разбирается мотивация, формальная постановка, алгоритм, ограничения и связь с остальными подходами. В конце — сравнение и практические рекомендации.

---

## Глава 1. Постановка задачи и базовая терминология

### 1.1. Что такое sparse training

Пусть $f(x; \theta)$ — нейронная сеть с параметрами $\theta \in \mathbb{R}^N$. Введём бинарную маску $m \in \{0, 1\}^N$ и определим эффективные параметры как $\tilde\theta = \theta \odot m$, где $\odot$ — поэлементное умножение.

Задача sparse training:

$$
\min_{\theta, m} \; \mathcal{L}\big(f(x; \theta \odot m)\big) \quad \text{при условии} \quad \|m\|_0 \leq k
$$

Принципиальное отличие от классического прунинга — оптимизация ведётся одновременно по $\theta$ и $m$ с самого начала обучения, а ограничение $\|m\|_0 \leq k$ соблюдается на каждом шаге, а не достигается в конце.

### 1.2. Sparsity ratio

Степень разреженности:

$$
s = 1 - \frac{\|m\|_0}{N}
$$

При $s = 0.9$ работает 10% параметров. Обычно $s$ задаётся пользователем как гиперпараметр и поддерживается константным на протяжении обучения (за исключением методов с фазой «отжига» — pruning schedule).

### 1.3. Распределение разреженности по слоям

Если общая sparsity $s$ задана, остаётся вопрос — как распределить веса между слоями. Два основных подхода:

- Uniform sparsity — одинаковый процент сохранённых весов в каждом слое. Простой, но игнорирует разную чувствительность слоёв.
- Erdős–Rényi (ER) и ERK (Erdős–Rényi-Kernel) — sparsity слоя обратно пропорциональна размеру: маленькие слои оставляются плотнее, большие — реже. ERK дополнительно учитывает размер ядра в свёртках. Формула для свёрточного слоя с $n^{l-1}$ входными и $n^l$ выходными каналами и ядром $w \times h$:

$$
p_{\text{keep}}^l \propto \frac{n^{l-1} + n^l + w + h}{n^{l-1} \cdot n^l \cdot w \cdot h}
$$

ERK эмпирически работает значительно лучше uniform; разница может составлять несколько процентов точности на ImageNet.

### 1.4. Категоризация динамических методов

Динамические методы можно разделить по двум осям.

По моменту фиксации маски:

- Pruning at initialization (PaI) — маска вычисляется один раз до обучения и фиксируется. Сюда относятся SNIP, GraSP, SynFlow.
- Dynamic Sparse Training (DST) — маска обновляется на протяжении всего обучения. Сюда относятся SET, RigL, Top-KAST, MEST.

По критерию выбора весов:

- На основе magnitude (SET, RigL частично).
- На основе градиентов (SNIP, RigL).
- На основе кривизны / произведения градиентов (GraSP).
- На основе потока через сеть (SynFlow).

---

## Глава 2. SET — Sparse Evolutionary Training

Mocanu et al., *Scalable Training of Artificial Neural Networks with Adaptive Sparse Connectivity Inspired by Network Science*, Nature Communications, 2018.

SET — исторически первый по-настоящему рабочий метод динамического прунинга. Идея настолько проста, что кажется почти наивной, но именно SET положил начало целому направлению.

### 2.1. Идея

Поддерживать заданную sparsity на протяжении обучения, периодически:

1. Удалять часть наименее значимых связей (по абсолютной величине весов).
2. Добавлять такое же количество новых случайных связей (взамен удалённых).

Сеть «эволюционирует»: плохие связи отмирают, новые случайные «мутируют», полезные закрепляются обучением.

### 2.2. Алгоритм

Параметры: общая sparsity $s$, частота обновления маски (обычно раз в эпоху), доля обновления $\zeta$ (типично 0.2–0.3).

1. Инициализация: создать случайную бинарную маску $m$ с заданной sparsity. Веса в маске инициализируются как обычно (например, He или Glorot).
2. Обучение в течение одной эпохи (стандартный SGD с маскированием).
3. Prune-фаза: в каждом слое $l$ удалить $\zeta \cdot \|m^l\|_0$ весов с наименьшим $|\theta_i|$. Это снижает sparsity локально.
4. Grow-фаза: случайно активировать столько же ранее замаскированных связей. Новые веса инициализируются нулём (или малыми случайными значениями в некоторых реализациях).
5. Повторить шаги 2–4 до конца обучения.
6. Финальная стадия: на последних эпохах prune-grow обычно отключается, чтобы веса успели стабилизироваться.

### 2.3. Что делает SET особенным

- Полностью покрывает обучение в режиме sparse — плотная сеть никогда не материализуется.
- Не требует вторых производных или дорогих оценок важности.
- Случайный grow удивительно хорошо работает на практике — на MNIST, CIFAR и небольших задачах достигает качества плотной сети при sparsity 90%+.

### 2.4. Ограничения

- На больших задачах (ImageNet, языковые модели) случайный grow проигрывает более информированным стратегиям (RigL).
- Sparsity распределяется uniform; нет принципа выбора между слоями.
- Гиперпараметр $\zeta$ требует тюнинга — слишком маленький замедляет адаптацию, слишком большой разрушает обучение.

### 2.5. Связь с дальнейшими работами

SET задал шаблон prune-and-grow, который повторяется в почти всех последующих DST-методах. Развитие шло по двум направлениям:

- Улучшение grow-критерия (RigL, GraNet) — заменить случайность на градиентные сигналы.
- Улучшение распределения sparsity по слоям (ERK, dynamic sparse reparameterization).

---

## Глава 3. SNIP — Single-shot Network Pruning at Initialization

Lee, Ajanthan & Torr, *SNIP: Single-shot Network Pruning based on Connection Sensitivity*, ICLR 2019.

SNIP — первая успешная попытка определить «полезные» связи до начала обучения. В терминах нашей таксономии — это pruning at initialization, не DST в чистом виде. Но в контексте динамического прунинга он принципиален: SNIP показал, что важность связи можно оценить градиентным сигналом ещё на инициализированной сети.

### 3.1. Идея

Чувствительность связи $i$ к её наличию можно оценить так: насколько изменится лосс, если эту связь *удалить*? Вместо явного вычисления для каждого веса (что потребовало бы $N$ forward-проходов), используется аппроксимация через градиент.

### 3.2. Формализация

Введём вспомогательные индикаторные переменные $c_i \in \{0, 1\}$ и перепишем сеть как $f(x; c \odot \theta)$. Тогда эффект удаления связи $i$:

$$
\Delta \mathcal{L}_i = \mathcal{L}(c \odot \theta) - \mathcal{L}((c - e_i) \odot \theta)
$$

где $e_i$ — стандартный базисный вектор. При непрерывной релаксации $c \in \mathbb{R}^N$, $c = 1$:

$$
\Delta \mathcal{L}_i \approx \frac{\partial \mathcal{L}}{\partial c_i}\bigg|_{c=1} = \frac{\partial \mathcal{L}}{\partial \theta_i} \cdot \theta_i
$$

Это и есть SNIP-saliency. Часто берётся абсолютное значение:

$$
s_i^{\text{SNIP}} = \left| \theta_i \cdot \frac{\partial \mathcal{L}}{\partial \theta_i} \right|
$$

### 3.3. Алгоритм

1. Инициализировать сеть стандартным способом (He / Glorot — важно).
2. Взять мини-батч обучающих данных.
3. Сделать один forward-backward проход, получить градиенты.
4. Посчитать saliency $s_i = |\theta_i \cdot g_i|$ для всех весов.
5. Глобально отсортировать и оставить top-$k$, где $k = (1 - s) N$.
6. Зафиксировать маску $m$.
7. Дальше — стандартное обучение в режиме sparse.

### 3.4. Интерпретация

SNIP-saliency можно понимать двояко. С одной стороны, это первый член разложения Тейлора лосса по $c_i$ — оценка чувствительности к удалению. С другой — это произведение веса и градиента, то есть «вклад параметра в текущее обновление». Большие $|\theta \cdot g|$ означают: вес большой и градиент по нему большой, значит, он активно участвует в формировании предсказаний.

### 3.5. Ограничения

- Маска фиксируется на инициализированной сети. Если выяснится, что какие-то «отрезанные» связи были бы важны после нескольких эпох обучения — поздно, обратно их не вернуть.
- Чувствительность к инициализации: SNIP опирается на конкретную $\theta_0$. Плохая инициализация → плохая маска.
- На очень высоких sparsity (>95%) SNIP может приводить к layer collapse — полному отрезанию какого-то слоя, что делает сеть необучаемой (эту проблему решит SynFlow, см. главу 6).

### 3.6. Сравнение с magnitude pruning at initialization

Magnitude pruning на инициализированной сети — слабый baseline: при He-инициализации $|\theta_i|$ почти равномерны, отбирать почти нечего. SNIP в этом смысле принципиально важнее: он использует информацию о задаче через градиент. Без этого pruning at initialization вообще не имел бы смысла.

---

## Глава 4. GraSP — Gradient Signal Preservation

Wang, Zhang & Grosse, *Picking Winning Tickets Before Training by Preserving Gradient Flow*, ICLR 2020.

GraSP — развитие идеи SNIP с более тонким критерием. Авторы заметили, что SNIP оптимизирует лосс в нулевом приближении, но игнорирует, как прунинг повлияет на *обучаемость* сети — на динамику градиентов в начале обучения.

### 4.1. Мотивация

После прунинга по SNIP сеть может иметь хорошие предсказания на инициализации, но плохо обучаться: градиентный поток через сеть нарушен. Идея GraSP — выбирать маску так, чтобы максимально сохранить норму градиента, то есть обучаемость.

### 4.2. Формализация

Сохранять следует не лосс, а градиентный поток. Рассмотрим, как удаление веса влияет на норму градиента $\|g\|^2 = g^\top g$.

Через разложение Тейлора по $\delta\theta$:

$$
\Delta(g^\top g) \approx 2 g^\top H \, \delta\theta
$$

где $H$ — гессиан лосса. При $\delta\theta_i = -\theta_i$ (удаление веса $i$):

$$
\Delta(g^\top g)_i \approx -2 (Hg)_i \cdot \theta_i
$$

Знак здесь принципиален: GraSP-saliency определяется как

$$
s_i^{\text{GraSP}} = -(Hg)_i \cdot \theta_i
$$

(минус, потому что мы хотим *сохранить* поток градиента, то есть удалять веса, удаление которых *увеличивает* норму градиента или уменьшает её меньше всего).

### 4.3. Как считать $Hg$ без явного гессиана

Произведение гессиана на вектор вычисляется через двойной backward без хранения матрицы:

$$
Hg = \nabla_\theta (g^\top g) / 2
$$

Это стандартный приём «Hessian-vector product» — стоит примерно как два обычных backward'а.

### 4.4. Алгоритм

1. Инициализировать сеть.
2. Прогнать мини-батч, посчитать градиент $g$.
3. Посчитать $Hg$ через двойной backward.
4. Посчитать $s_i = -(Hg)_i \cdot \theta_i$ для каждого веса.
5. Оставить top-$k$ по saliency, остальные обнулить.
6. Обучать sparse-сеть.

### 4.5. Сравнение с SNIP

| Аспект | SNIP | GraSP |
|---|---|---|
| Критерий | $\lvert\theta_i g_i\rvert$ | $-(Hg)_i \theta_i$ |
| Что сохраняет | значение лосса | норму градиента |
| Стоимость | 1 backward | 2 backward (HVP) |
| На высоких sparsity | склонен к layer collapse | устойчивее |

На практике GraSP обычно даёт лучшие результаты при экстремальных sparsity, но разница на умеренных sparsity (50–80%) невелика.

### 4.6. Ограничения

Те же, что у SNIP: маска фиксируется до обучения, не адаптируется. Кроме того, HVP вносит дополнительный шум — на маленьких батчах оценка $Hg$ может быть нестабильной.

---

## Глава 5. RigL — Rigging the Lottery

Evci et al., *Rigging the Lottery: Making All Tickets Winners*, ICML 2020.

RigL — на момент выхода state-of-the-art в динамическом прунинге, и до сих пор сильный baseline. Его можно считать «правильным» развитием SET: prune-and-grow, но grow по информированному критерию.

### 5.1. Идея

В SET добавляются случайные связи. RigL заменяет случайность на градиентный сигнал: добавляются те связи, у которых сейчас *самый большой градиент по модулю* среди замаскированных. Логика: большой градиент означает, что эта связь, будь она активна, сильно влияла бы на лосс — значит, есть смысл её активировать.

### 5.2. Алгоритм

Параметры: sparsity $s$, частота обновления $\Delta T$ (например, каждые 100 шагов), начальная доля обновления $\alpha_0$ (типично 0.3), расписание затухания $\alpha_t$.

1. Инициализация: маска по ERK, веса по стандартной инициализации.
2. Стандартное обучение с маскированием.
3. Каждые $\Delta T$ шагов:
   - Prune: в каждом слое удалить $\alpha_t$-долю активных весов с наименьшим $|\theta_i|$.
   - Grow: в том же слое активировать столько же замаскированных весов с наибольшим $|\partial \mathcal{L}/\partial \theta_i|$. Новые веса инициализируются нулём.
4. Затухание $\alpha_t$ по косинусу:

$$
\alpha_t = \frac{\alpha_0}{2}\left(1 + \cos\frac{\pi t}{T_{\text{end}}}\right)
$$

5. После $T_{\text{end}}$ маска фиксируется, обучение идёт без обновлений.

### 5.3. Почему grow по градиенту работает

Чтобы посчитать градиент по замаскированному весу $\theta_i$ (т.е. по весу, который сейчас обнулён), достаточно стандартного backward: формально градиент существует и для нулевых весов, просто он не используется для обновления, пока маска == 0. Один лишний шаг — это сохранить значения градиентов для замаскированных позиций при backward-проходе на момент обновления маски.

Это означает, что RigL получает информацию о «потенциально полезных» связях практически бесплатно — никаких дополнительных проходов, в отличие от GraSP.

### 5.4. Распределение sparsity и роль ERK

В оригинальной работе RigL с uniform sparsity работает заметно хуже, чем с ERK. Это типичный паттерн в DST: правильное распределение разреженности по слоям может дать больше, чем умный prune-grow критерий.

### 5.5. Результаты

RigL — первый sparse-training метод, который на ResNet-50 / ImageNet при 80% sparsity достигает точности плотной модели, не используя плотную сеть ни на одном этапе. При 90% sparsity отставание небольшое (порядка 1–2% top-1). Это была принципиальная веха.

### 5.6. Ограничения

- Гиперпараметры: $\Delta T$, $\alpha_0$, расписание затухания — требуют тюнинга.
- Не масштабируется тривиально на трансформеры и LLM: там структура важности весов сложнее, и magnitude-prune часть может удалять стратегически важные веса.

---

## Глава 6. SynFlow — Synaptic Flow Pruning

Tanaka et al., *Pruning neural networks without any data by iteratively conserving synaptic flow*, NeurIPS 2020.

SynFlow — особый зверь: pruning at initialization без использования данных вообще. Решает проблему layer collapse в SNIP/GraSP теоретически чисто.

### 6.1. Проблема layer collapse

При очень высокой sparsity SNIP может полностью отрезать какой-то слой: все его веса попадают в нижний хвост saliency, удаляются, и сеть теряет способность пропускать сигнал. Layer collapse — это резкое падение качества из-за нарушения связности, а не из-за общей нехватки параметров.

Авторы SynFlow доказали: любой пер-весовой критерий saliency, удовлетворяющий определённым условиям, гарантированно избегает layer collapse, если применяется итеративно (а не one-shot).

### 6.2. Synaptic flow

Определяется как:

$$
\mathcal{R}_{\text{SF}}(\theta) = \mathbf{1}^\top \left(\prod_{l=1}^{L} |W^l|\right) \mathbf{1}
$$

где $|W^l|$ — матрица модулей весов слоя $l$, $\mathbf{1}$ — вектор из единиц. Содержательно — это сумма по всем путям от входа к выходу через сеть с заменой $W^l \to |W^l|$.

Saliency веса $\theta_i$:

$$
s_i^{\text{SF}} = \left| \frac{\partial \mathcal{R}_{\text{SF}}}{\partial \theta_i} \cdot \theta_i \right|
$$

Интерпретация: SynFlow оценивает, насколько данный вес участвует в потоках от входа к выходу. Веса, через которые проходит много путей, важны; те, через которые ничего не идёт, удаляются.

### 6.3. Алгоритм

1. Заменить все веса на их модули.
2. Прогнать вектор единиц через сеть, посчитать $\mathcal{R}_{\text{SF}}$.
3. Через backward посчитать $\partial \mathcal{R}_{\text{SF}}/\partial \theta_i$.
4. Удалить $\Delta s$ долю весов с наименьшим saliency.
5. Повторить $T$ итераций, постепенно достигая целевой sparsity.

Никаких данных, никаких градиентов лосса. Только структура сети.

### 6.4. Что показывает SynFlow

- Sparsity-aware теория: при правильно сформулированном критерии layer collapse невозможен.
- Конкурентоспособные результаты с SNIP/GraSP на высоких sparsity, без данных.
- Прекрасно работает как «инициализация маски» для последующего DST.

### 6.5. Ограничения

Без данных метод не может учесть особенности конкретной задачи. На умеренных sparsity SNIP/GraSP/RigL чаще выигрывают, потому что используют сигнал задачи. SynFlow ценен в первую очередь как теоретический инструмент и решение проблемы layer collapse.

---

## Глава 7. Top-KAST и другие современные подходы

### 7.1. Top-KAST

Jayakumar et al., *Top-KAST: Top-K Always Sparse Training*, NeurIPS 2020.

Отличие от RigL: маска определяется *на ходу* без явных prune-grow шагов. На каждом forward-проходе:

- Активны top-$K$ весов по модулю (forward sparse).
- Backward проходит через все веса (или через расширенный набор top-$K'$, $K' > K$), что позволяет градиентам обновлять и «спящие» веса.

Это делает выбор активных весов непрерывно адаптивным: вес, который вырос по модулю выше порога, автоматически попадает в активную маску в следующем шаге. Маска не нуждается в явных эпохах обновления.

Преимущество: меньше гиперпараметров. Недостаток: вычислительная стоимость выше RigL, потому что backward плотнее forward'а.

### 7.2. MEST

Yuan et al., *MEST: Accurate and Fast Memory-Economic Sparse Training Framework for the Edge*, NeurIPS 2021.

Развитие RigL для edge-устройств. Основные идеи:

- Memory-economic: маска и градиенты хранятся в sparse-формате на всём цикле.
- EM (elastic mutation) — softer prune-grow: вместо жёсткого удаления используется evidence-based порог.
- Saliency: совмещает magnitude и градиент в единый скор $s_i = |\theta_i| + \lambda |\partial \mathcal{L}/\partial \theta_i|$.

### 7.3. GraNet

Liu et al., *Sparse Training via Boosting Pruning Plasticity with Neuroregeneration*, NeurIPS 2021.

Идея: вместо одной фиксированной sparsity использовать «отжиг» — начать с плотной сети и постепенно увеличивать sparsity до целевой, продолжая prune-grow на всём пути. Метод dynamic в обоих смыслах: меняется и маска, и общий уровень разреженности.

### 7.4. Powerpropagation

Schwarz et al., *Powerpropagation: A Sparsity Inducing Weight Reparameterisation*, NeurIPS 2021.

Не «прунинг» в строгом смысле, но связанная идея. Веса репараметризуются как $\theta = w \cdot |w|^{\alpha - 1}$ для $\alpha > 1$. Это сжимает маленькие веса к нулю быстрее, чем линейная динамика SGD. После обучения многие веса оказываются эффективно нулевыми и могут быть формально занулены без потерь.

---

## Глава 8. Сравнительный анализ

### 8.1. Сводная таблица

| Метод | Тип | Маска фиксируется | Критерий prune | Критерий grow | Требует данных |
|---|---|---|---|---|---|
| SET | DST | нет | magnitude | random | да |
| SNIP | PaI | до обучения | $\lvert\theta g\rvert$ | — | да |
| GraSP | PaI | до обучения | $-(Hg)\theta$ | — | да |
| SynFlow | PaI | до обучения | path flow | — | нет |
| RigL | DST | нет | magnitude | $\lvert g\rvert$ | да |
| Top-KAST | DST | непрерывно | magnitude | автоматический | да |
| MEST | DST | нет | magnitude+grad | grad | да |

### 8.2. Что выбирать в зависимости от задачи

- Если нужно максимальное качество и есть бюджет: RigL с ERK при умеренной sparsity (50–80%) — крепкая отправная точка. Top-KAST даёт сравнимые результаты с меньшим количеством гиперпараметров.
- Если ограничен бюджет памяти на обучение: MEST или RigL — обе категорически избегают плотной сети.
- Если нужен дешёвый «one-shot» подход без обучения с нуля: SNIP / GraSP. SNIP проще, GraSP лучше на высоких sparsity.
- Если sparsity экстремальна (>95%): SynFlow для инициализации маски + RigL поверх неё. Так избегается layer collapse и сохраняется адаптивность.
- Если плотная сеть уже обучена: классический iterative magnitude pruning с дообучением — простой и сильный подход, его не стоит сбрасывать.

### 8.3. Ловушки бенчмаркинга

При сравнении методов важно учитывать:

- Бюджет обучения (FLOPs / эпохи). DST методы могут требовать больше эпох для сходимости — сравнение с плотной сетью на одинаковом числе эпох может быть нечестным.
- Распределение sparsity. Один и тот же метод с uniform и с ERK может различаться на 2–3% точности. Сравнение методов с разным распределением — некорректно.
- Реальное ускорение vs теоретическое. На стандартном железе unstructured sparse-обучение часто *медленнее* плотного из-за неэффективности sparse-операций.

---

## Глава 9. Теоретические перспективы

DST породил несколько интересных теоретических вопросов:

- Lottery Ticket vs DST. Frankle & Carbin показали, что внутри плотной сети существуют выигрышные подсети. DST утверждает, что искать их через обучение плотной сети необязательно — их можно «вырастить» с нуля. Эмпирически оба подхода дают сравнимые результаты, но механизмы разные.
- Сходимость sparse training. Теоретических гарантий сходимости для DST почти нет; известно лишь, что при достаточно медленных обновлениях маски метод асимптотически эквивалентен прунингу обученной сети.
- Роль over-parameterization. DST бросает вызов представлению о необходимости избыточных параметров для обучения. Если sparse-сеть из 10% весов обучается до того же качества, то роль «лишних» 90% — не обучение, а навигация по ландшафту лосса. Это активная область исследований.

---

## Заключение

Динамический прунинг прошёл путь от наивной случайной эволюции SET до тонко настроенных методов вроде RigL и Top-KAST. Основные уроки:

- Маска не обязана быть фиксированной — её эволюция во время обучения принципиально полезна.
- Информированный grow (по градиенту) значительно лучше случайного.
- Распределение sparsity по слоям (ERK) даёт улучшение, сравнимое с заменой критерия prune-grow.
- Pruning at initialization (SNIP, GraSP, SynFlow) — компромисс между простотой и качеством; на экстремальных sparsity SynFlow незаменим для избегания layer collapse.
- Реальное ускорение всё ещё ограничено аппаратной поддержкой sparse-операций. Без неё DST — это в основном память-эффективность, а не скорость.

В контексте больших моделей (LLM, диффузионные модели) DST пока менее зрел: методы вроде SparseGPT работают post-hoc и доминируют по практической применимости. Но направление активное, и адаптация RigL-подобных подходов к трансформерам — одна из открытых задач.

---

## Литература

- Mocanu et al. *Scalable Training of Artificial Neural Networks with Adaptive Sparse Connectivity*. Nature Communications, 2018. (SET)
- Lee, Ajanthan, Torr. *SNIP: Single-shot Network Pruning based on Connection Sensitivity*. ICLR 2019.
- Wang, Zhang, Grosse. *Picking Winning Tickets Before Training by Preserving Gradient Flow*. ICLR 2020. (GraSP)
- Evci et al. *Rigging the Lottery: Making All Tickets Winners*. ICML 2020. (RigL)
- Tanaka et al. *Pruning neural networks without any data by iteratively conserving synaptic flow*. NeurIPS 2020. (SynFlow)
- Jayakumar et al. *Top-KAST: Top-K Always Sparse Training*. NeurIPS 2020.
- Yuan et al. *MEST: Accurate and Fast Memory-Economic Sparse Training*. NeurIPS 2021.
- Liu et al. *Sparse Training via Boosting Pruning Plasticity with Neuroregeneration*. NeurIPS 2021. (GraNet)
- Schwarz et al. *Powerpropagation*. NeurIPS 2021.
- Frankle & Carbin. *The Lottery Ticket Hypothesis*. ICLR 2019.
- Hoefler et al. *Sparsity in Deep Learning*. JMLR 2021. — расширенный survey по теме.